In [20]:
from bs4 import BeautifulSoup
import re
import emoji
import string
import contractions
from symspellpy import SymSpell
import pkg_resources
import nltk
    
resources = [
        'punkt',
        'punkt_tab',
        'stopwords',
        'wordnet',
        'omw-1.4',
        'averaged_perceptron_tagger_eng'
    ]
    
for resource in resources:
        nltk.download(resource)
    
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.util import ngrams
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       d

In [21]:
def process_data(text):
       
    text = BeautifulSoup(text, "html.parser").text
          
    text = contractions.fix(text)
     
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    text = re.sub(r'\S+@\S+', '', text)
    
    text = re.sub(r'\+?\d[\d\s-]{8,}\d', '', text)
       
    text = emoji.replace_emoji(text, replace='')
    
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    text = text.lower()
    
    dictionary_path = pkg_resources.resource_filename(
        "symspellpy",
        "frequency_dictionary_en_82_765.txt"
    )
    
    # Create object
    sym_spell = SymSpell(max_dictionary_edit_distance=2)
    
    # Load dictionary
    dictionary_path = pkg_resources.resource_filename(
        "symspellpy",
        "frequency_dictionary_en_82_765.txt"
    )
    
    sym_spell.load_dictionary(
        dictionary_path,
        term_index=0,
        count_index=1
    )
    
    
    suggestions = sym_spell.lookup_compound(
        text,
        max_edit_distance=2
    )
    
    text=suggestions[0].term
    
    tokens = word_tokenize(text)
    
    stop_words = set(stopwords.words('english'))
    
    tokens = [word for word in tokens if word not in stop_words]
    
    lemmatizer = WordNetLemmatizer()
    
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    text = " ".join(tokens)
    
    return text

In [22]:
folder_path = r"C:\Users\Think\Desktop\anacondaaa\nlp\NLP Models\plagiarism"

documents = {}

for file in os.listdir(folder_path):
    if file.endswith(".txt"):
        file_path = os.path.join(folder_path, file)

        with open(file_path, "r", encoding="utf-8") as f:
            documents[file] = f.read()

print("Total Documents:", len(documents))

Total Documents: 2


In [23]:
processed_documents = {}
for filename, text in documents.items():
    processed_documents[filename] = process_data(text)

filename = list(documents.keys())[0]



In [24]:
print(len(processed_documents))

2


In [25]:
model = SentenceTransformer('all-MiniLM-L6-v2')
filenames = list(processed_documents.keys())

texts = list(processed_documents.values())
embeddings = model.encode(texts)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(2, 384)


In [27]:
similarity_matrix = cosine_similarity(embeddings)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=filenames,
    columns=filenames
)

similarity_df.round(4)

,doc_001.txt,doc_002.txt
doc_001.txt,1.0000,0.9347
doc_002.txt,0.9347,1.0000


In [32]:
folder_path = r"C:\Users\Think\Desktop\anacondaaa\nlp\NLP Models\plagiarism_corpus_20_docs"

documents = {}

for file in os.listdir(folder_path):
    if file.endswith(".txt"):
        file_path = os.path.join(folder_path, file)

        with open(file_path, "r", encoding="utf-8") as f:
            documents[file] = f.read()

print("Total Documents:", len(documents))

Total Documents: 20


In [35]:
processed_documents = {}
for filename, text in documents.items():
    processed_documents[filename] = process_data(text)

filename = list(documents.keys())

print(len(processed_documents))
len(filename)

20


20

In [38]:
model = SentenceTransformer('all-MiniLM-L6-v2')
filenames = list(processed_documents.keys())

texts = list(processed_documents.values())
embeddings = model.encode(texts)
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(20, 384)


In [39]:
similarity_matrix = cosine_similarity(embeddings)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=filenames,
    columns=filenames
)

similarity_df.round(4)

,doc_001.txt,doc_002.txt,doc_003.txt,doc_004.txt,doc_005.txt,doc_006.txt,doc_007.txt,doc_008.txt,doc_009.txt,doc_010.txt,doc_011.txt,doc_012.txt,doc_013.txt,doc_014.txt,doc_015.txt,doc_016.txt,doc_017.txt,doc_018.txt,doc_019.txt,doc_020.txt
doc_001.txt,1.0000,0.8213,0.5366,0.7243,0.7992,1.0000,0.8213,0.5366,0.7243,0.7992,0.8339,0.6619,0.3580,0.5331,0.5998,0.1039,0.1533,0.0454,0.0888,0.1320
doc_002.txt,0.8213,1.0000,0.5021,0.6535,0.7491,0.8213,1.0000,0.5021,0.6535,0.7491,0.6754,0.8309,0.3474,0.4974,0.5701,0.1019,0.1046,0.0177,0.0553,0.0712
doc_003.txt,0.5366,0.5021,1.0000,0.5197,0.5707,0.5366,0.5021,1.0000,0.5197,0.5707,0.3721,0.3753,0.8481,0.3664,0.3927,0.0943,0.0635,0.1724,0.0205,-0.0488
doc_004.txt,0.7243,0.6535,0.5197,1.0000,0.7344,0.7243,0.6535,0.5197,1.0000,0.7344,0.5555,0.5078,0.3387,0.8151,0.5521,0.0697,0.0509,0.0711,0.1055,0.1162
doc_005.txt,0.7992,0.7491,0.5707,0.7344,1.0000,0.7992,0.7491,0.5707,0.7344,1.0000,0.6696,0.6224,0.3973,0.5686,0.8435,0.0647,0.1479,0.0698,0.1246,0.0818
doc_006.txt,1.0000,0.8213,0.5366,0.7243,0.7992,1.0000,0.8213,0.5366,0.7243,0.7992,0.8339,0.6619,0.3580,0.5331,0.5998,0.1039,0.1533,0.0454,0.0888,0.1320
doc_007.txt,0.8213,1.0000,0.5021,0.6535,0.7491,0.8213,1.0000,0.5021,0.6535,0.7491,0.6754,0.8309,0.3474,0.4974,0.5701,0.1019,0.1046,0.0177,0.0553,0.0712
doc_008.txt,0.5366,0.5021,1.0000,0.5197,0.5707,0.5366,0.5021,1.0000,0.5197,0.5707,0.3721,0.3753,0.8481,0.3664,0.3927,0.0943,0.0635,0.1724,0.0205,-0.0488
doc_009.txt,0.7243,0.6535,0.5197,1.0000,0.7344,0.7243,0.6535,0.5197,1.0000,0.7344,0.5555,0.5078,0.3387,0.8151,0.5521,0.0697,0.0509,0.0711,0.1055,0.1162
doc_010.txt,0.7992,0.7491,0.5707,0.7344,1.0000,0.7992,0.7491,0.5707,0.7344,1.0000,0.6696,0.6224,0.3973,0.5686,0.8435,0.0647,0.1479,0.0698,0.1246,0.0818
